## Imports

In [1]:
import pandas as pd
import os
import numpy as np
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv, find_dotenv
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize

C:\Users\PC\anaconda3\envs\vectordb_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Data

In [2]:
files = pd.read_csv("course_descriptions.csv", encoding="ANSI")

## Create Course Descriptions

In [3]:
def create_course_description(row):
    return f'''The course name is {row["course_name"]}, the slug is {row["course_slug"]},
            the technology is {row["course_technology"]} and the course topic is {row["course_topic"]}'''

pd.set_option('display.max_rows', 106)
files['course_description_new'] = files.apply(create_course_description, axis=1)
print(files["course_description_new"])

0      The course name is Introduction to Tableau, th...
1      The course name is The Complete Data Visualiza...
2      The course name is Introduction to R Programmi...
3      The course name is Data Preprocessing with Num...
4      The course name is Introduction to Data and Da...
5      The course name is Data Cleaning and Preproces...
6      The course name is Introduction to Business An...
7      The course name is Data Analysis with Excel Pi...
8      The course name is SQL, the slug is sql,\n    ...
9      The course name is Credit Risk Modeling in Pyt...
10     The course name is Python Programmer Bootcamp,...
11     The course name is SQL + Tableau + Python, the...
12     The course name is Introduction to Jupyter, th...
13     The course name is Statistics, the slug is sta...
14     The course name is Mathematics, the slug is ma...
15     The course name is Introduction to Excel, the ...
16     The course name is Probability, the slug is pr...
17     The course name is Start

In [4]:
def get_combined_text(row):
    return ' '.join([str(row[field]) for field in ['course_description', 'course_description_new', 'course_description_short']])

files['combined_text'] = files.apply(get_combined_text, axis=1)

## Load Environment Variables

In [5]:
%load_ext dotenv
%dotenv
load_dotenv(find_dotenv(), override=True)

True

## Pinecone Setup

Instead of creating one index per algorithm (which hits the free-plan limit of 5 indexes),
we use **two shared indexes** — one per vector dimension — and separate each algorithm
using **namespaces**. A namespace is a logical partition inside a single index;
queries within a namespace only search vectors upserted to that namespace.

| Index | Dimension | Algorithms |
|---|---|---|
| `index-768` | 768 | BoW, TF-IDF, BERT, ELMo |
| `index-384` | 384 | Word2Vec |

In [6]:
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))

In [7]:
def get_or_create_index(index_name, dimension, metric="cosine"):
    existing = [idx.name for idx in pc.list_indexes()]
    if index_name not in existing:
        pc.create_index(
            name=index_name,
            dimension=dimension,
            metric=metric,
            spec=ServerlessSpec(cloud="aws", region="us-east-1")
        )
        print(f"Created '{index_name}' (dim={dimension}).")
    else:
        print(f"'{index_name}' already exists — reusing it.")
    return pc.Index(index_name)

# Two shared indexes — one per vector dimension
index_768 = get_or_create_index("index-768", dimension=768)
index_384 = get_or_create_index("index-384", dimension=384)

Created 'index-768' (dim=768).
Created 'index-384' (dim=384).


In [8]:
def upsert_embeddings(index, ids, embeddings, namespace):
    vectors = [
        (str(id_), emb.tolist() if hasattr(emb, 'tolist') else list(emb))
        for id_, emb in zip(ids, embeddings)
    ]
    index.upsert(vectors=vectors, namespace=namespace)
    print(f"Upserted {len(vectors)} vectors to namespace='{namespace}'.")

def run_query(index, query_embedding, namespace, top_k=12, score_threshold=0.3):
    query_vec = query_embedding.tolist() if hasattr(query_embedding, 'tolist') else list(query_embedding)
    results = index.query(vector=[query_vec], top_k=top_k, include_values=False, namespace=namespace)
    return [m for m in results["matches"] if m["score"] >= score_threshold]

def print_results(query, matches):
    print(f"\nQuery: '{query}'")
    if matches:
        for m in matches:
            print(f"  {m['id']} — score: {m['score']:.4f}")
    else:
        print("  No matches above threshold.")

queries = ["clustering", "data science", "python", "finance"]

## 1. Bag of Words (BoW)

Represents text as word occurrence counts. Simple but has no semantic understanding — "car" and "automobile" are treated as completely unrelated.

Uses namespace `bow` inside `index-768`.

In [9]:
BOW_FEATURES = 768

bow_vectorizer = CountVectorizer(max_features=BOW_FEATURES)
bow_matrix = bow_vectorizer.fit_transform(files['combined_text']).toarray()
bow_matrix = normalize(bow_matrix).astype(np.float32)

upsert_embeddings(index_768, files["course_name"], bow_matrix, namespace="bow")

Upserted 106 vectors to namespace='bow'.


In [10]:
# --- BoW Query ---
for query in queries:
    query_vec = bow_vectorizer.transform([query]).toarray()
    query_vec = normalize(query_vec).astype(np.float32)[0]
    matches = run_query(index_768, query_vec, namespace="bow")
    print_results(query, matches)


Query: 'clustering'
  No matches above threshold.

Query: 'data science'
  Python for Social Media Analytics — score: 0.4743
  Data Literacy — score: 0.3588
  Introduction to Data and Data Science — score: 0.6146
  Starting a Career in Data Science: Project Portfolio, Resume, and Interview Process — score: 0.3849
  Power Query and Data Modeling — score: 0.3400
  ChatGPT for Data Science — score: 0.5240
  Mathematics — score: 0.3387
  Product Management for AI & Data Science — score: 0.3375
  Data-Driven Business Growth — score: 0.3349
  How to Think Like a Data Scientist to Become One — score: 0.3083

Query: 'python'
  Intermediate Python Programming — score: 0.3802
  Python Programmer Bootcamp — score: 0.3605
  Introduction to Python — score: 0.3511
  Machine Learning in Python — score: 0.3215
  Python for Finance — score: 0.3187
  Customer Analytics in Python — score: 0.3023

Query: 'finance'
  Who Does What in Finance — score: 0.4113


## 2. TF-IDF (Term Frequency–Inverse Document Frequency)

Improves on BoW by weighting words by how unique they are across all documents. Common words like "the" get lower scores; rare meaningful words score higher.

Uses namespace `tfidf` inside `index-768`.

In [11]:
TFIDF_FEATURES = 768

tfidf_vectorizer = TfidfVectorizer(max_features=TFIDF_FEATURES)
tfidf_matrix = tfidf_vectorizer.fit_transform(files['combined_text']).toarray().astype(np.float32)

upsert_embeddings(index_768, files["course_name"], tfidf_matrix, namespace="tfidf")

Upserted 106 vectors to namespace='tfidf'.


In [12]:
# --- TF-IDF Query ---
for query in queries:
    query_vec = tfidf_vectorizer.transform([query]).toarray().astype(np.float32)[0]
    matches = run_query(index_768, query_vec, namespace="tfidf")
    print_results(query, matches)


Query: 'clustering'
  Machine Learning in Python — score: 0.3139

Query: 'data science'
  Introduction to Data and Data Science — score: 0.6161
  ChatGPT for Data Science — score: 0.4109
  Python for Social Media Analytics — score: 0.3193
  Mathematics — score: 0.3193
  Product Management for AI & Data Science — score: 0.3060

Query: 'python'
  Introduction to Python — score: 0.4324
  Intermediate Python Programming — score: 0.4314
  Python Programmer Bootcamp — score: 0.4144
  Python for Finance — score: 0.3796
  Machine Learning in Python — score: 0.3542
  Web Scraping and API Fundamentals in Python — score: 0.3084

Query: 'finance'
  Who Does What in Finance — score: 0.6019


## 3. Word2Vec (approximated via averaged token embeddings)

Word2Vec maps individual words to vectors. For sentences, word vectors are averaged — which loses sentence-level context. We approximate this by averaging token embeddings from `all-MiniLM-L6-v2` without CLS pooling.

Uses namespace `w2v` inside `index-384` (384-dim model).

In [13]:
w2v_model = SentenceTransformer('all-MiniLM-L6-v2')

def word2vec_style_encode(texts, model):
    embeddings = []
    for text in texts:
        token_embeddings = model.encode(text, output_value='token_embeddings', show_progress_bar=False)
        avg_embedding = token_embeddings.mean(axis=0)
        embeddings.append(avg_embedding)
    return np.array(embeddings)

w2v_embeddings = word2vec_style_encode(files['combined_text'].tolist(), w2v_model)
w2v_embeddings = normalize(w2v_embeddings).astype(np.float32)

upsert_embeddings(index_384, files["course_name"], w2v_embeddings, namespace="w2v")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4545.55it/s]


Upserted 106 vectors to namespace='w2v'.


In [14]:
# --- Word2Vec-style Query ---
for query in queries:
    query_vec = word2vec_style_encode([query], w2v_model)[0]
    query_vec = normalize([query_vec])[0].astype(np.float32)
    matches = run_query(index_384, query_vec, namespace="w2v")
    print_results(query, matches)


Query: 'clustering'
  Machine Learning in Excel — score: 0.3550
  Machine Learning with K-Nearest Neighbors — score: 0.3141

Query: 'data science'
  Time Series Analysis with Python — score: 0.5736
  Mathematics — score: 0.5364
  Starting a Career in Data Science: Project Portfolio, Resume, and Interview Process — score: 0.5321
  Python for Social Media Analytics — score: 0.5346
  Deep Learning with TensorFlow — score: 0.5438
  Probability — score: 0.5634
  Introduction to Data and Data Science — score: 0.6727
  Data Strategy — score: 0.5321
  Data Literacy — score: 0.5213
  Deep Learning with TensorFlow 2 — score: 0.5093
  Statistics — score: 0.5061
  SQL — score: 0.5023

Query: 'python'
  Introduction to Jupyter — score: 0.5684
  Introduction to Python — score: 0.6181
  Intermediate Python Programming — score: 0.5770
  Python Programmer Bootcamp — score: 0.5580
  Machine Learning in Python — score: 0.5367
  Python for Finance — score: 0.5238
  Data Preprocessing with NumPy — score: 

## 4. BERT (`all-mpnet-base-v2`)

A transformer model producing context-aware embeddings. "bank" in "river bank" vs. "bank account" gets different vectors. `all-mpnet-base-v2` is one of the strongest general-purpose sentence embedding models on sbert.net.

Uses namespace `bert` inside `index-768`.

In [15]:
bert_model = SentenceTransformer('all-mpnet-base-v2')  # 768 dims

bert_embeddings = np.array([
    bert_model.encode(text, show_progress_bar=False)
    for text in files['combined_text']
]).astype(np.float32)

upsert_embeddings(index_768, files["course_name"], bert_embeddings, namespace="bert")

C:\Users\PC\anaconda3\envs\vectordb_env\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PC\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6038.90it/s]


Upserted 106 vectors to namespace='bert'.


In [16]:
# --- BERT Query ---
for query in queries:
    query_vec = bert_model.encode(query, show_progress_bar=False).astype(np.float32)
    matches = run_query(index_768, query_vec, namespace="bert")
    print_results(query, matches)


Query: 'clustering'
  Machine Learning with K-Nearest Neighbors — score: 0.4493
  Machine Learning in Excel — score: 0.4107
  Machine Learning with Naive Bayes — score: 0.3958
  Machine Learning in Python — score: 0.3917
  Machine Learning with Support Vector Machines — score: 0.3654
  The Machine Learning Process A-Z — score: 0.3489
  Introduction to Data and Data Science — score: 0.3440
  Linear Algebra and Feature Selection — score: 0.3419
  Data Preprocessing with NumPy — score: 0.3399
  Machine Learning with Decision Trees and Random Forests — score: 0.3387
  Data Cleaning and Preprocessing with pandas — score: 0.3249
  Machine Learning Deep Dive: Business Applications and Coding Walkthroughs — score: 0.3225

Query: 'data science'
  Working with Text Files in Python — score: 0.5386
  Introduction to Data and Data Science — score: 0.6625
  Statistics — score: 0.5754
  How to Think Like a Data Scientist to Become One — score: 0.5720
  Data Strategy — score: 0.5478
  Mathematics — s

## 5. ELMo-style (`all-distilroberta-v1`)

ELMo introduced context-aware embeddings using bidirectional LSTMs, predating BERT. We approximate it here using `all-distilroberta-v1` — a distilled RoBERTa model that is lighter than BERT while still being context-aware.

Uses namespace `elmo` inside `index-768`.

In [17]:
elmo_model = SentenceTransformer('all-distilroberta-v1')  # 768 dims

elmo_embeddings = np.array([
    elmo_model.encode(text, show_progress_bar=False)
    for text in files['combined_text']
]).astype(np.float32)

upsert_embeddings(index_768, files["course_name"], elmo_embeddings, namespace="elmo")

C:\Users\PC\anaconda3\envs\vectordb_env\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PC\.cache\huggingface\hub\models--sentence-transformers--all-distilroberta-v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5954.53it/s]


Upserted 106 vectors to namespace='elmo'.


In [18]:
# --- ELMo-style Query ---
for query in queries:
    query_vec = elmo_model.encode(query, show_progress_bar=False).astype(np.float32)
    matches = run_query(index_768, query_vec, namespace="elmo")
    print_results(query, matches)


Query: 'clustering'
  No matches above threshold.

Query: 'data science'
  Probability — score: 0.6344
  Statistics — score: 0.5659
  Time Series Analysis with Python — score: 0.4965
  How to Think Like a Data Scientist to Become One — score: 0.5854
  Introduction to Data and Data Science — score: 0.5392
  Starting a Career in Data Science: Project Portfolio, Resume, and Interview Process — score: 0.5580
  The Machine Learning Process A-Z — score: 0.4624
  Python for Social Media Analytics — score: 0.4340
  Data Literacy — score: 0.4295
  Data Strategy — score: 0.4270
  Mathematics — score: 0.4010
  Introduction to Python — score: 0.3945

Query: 'python'
  Python for Finance — score: 0.4339
  Introduction to Python — score: 0.5808
  Working with Text Files in Python — score: 0.4508
  Web Scraping and API Fundamentals in Python — score: 0.4067
  Intermediate Python Programming — score: 0.5242
  Machine Learning in Python — score: 0.4184
  Python Programmer Bootcamp — score: 0.5824
  Da

## Comparison Summary

In [19]:
models_to_compare = {
    "BoW":     (index_768, "bow",   lambda q: normalize(bow_vectorizer.transform([q]).toarray().astype(np.float32))[0]),
    "TF-IDF":  (index_768, "tfidf", lambda q: tfidf_vectorizer.transform([q]).toarray().astype(np.float32)[0]),
    "Word2Vec":(index_384, "w2v",   lambda q: normalize(word2vec_style_encode([q], w2v_model))[0].astype(np.float32)),
    "BERT":    (index_768, "bert",  lambda q: bert_model.encode(q, show_progress_bar=False).astype(np.float32)),
    "ELMo":    (index_768, "elmo",  lambda q: elmo_model.encode(q, show_progress_bar=False).astype(np.float32)),
}

test_query = "clustering"

print(f"{'='*60}")
print(f"Query: '{test_query}'")
print(f"{'='*60}")

for model_name, (index, namespace, encode_fn) in models_to_compare.items():
    query_vec = encode_fn(test_query)
    matches = run_query(index, query_vec, namespace=namespace, score_threshold=0.0, top_k=5)
    print(f"\n[{model_name}]")
    for m in matches:
        print(f"  {m['id']:<45} score: {m['score']:.4f}")

Query: 'clustering'

[BoW]
  Machine Learning in Python                    score: 0.1283
  Machine Learning in Excel                     score: 0.1255
  Who Does What in Finance                      score: 0.0000
  Fashion Analytics with Tableau                score: 0.0000
  Sign-Up Flow Optimization Analysis with SQL and Tableau score: 0.0000

[TF-IDF]
  Machine Learning in Python                    score: 0.3139
  Machine Learning in Excel                     score: 0.2767
  Sign-Up Flow Optimization Analysis with SQL and Tableau score: 0.0000
  Multiples Valuation                           score: 0.0000
  Web Scraping and API Fundamentals in Python   score: 0.0000

[Word2Vec]
  Machine Learning in Excel                     score: 0.3550
  Machine Learning with K-Nearest Neighbors     score: 0.3141
  Machine Learning in Python                    score: 0.2830
  Customer Churn Analysis with SQL and Tableau  score: 0.2813
  Growth Analysis with SQL, Python, and Tableau   score: 0.2597